# 历史辅助笔记
此 notebook 对应 2026-10-16 的三划分 top-5 误报候选筛选，不适用于当前六套全量 candidates 数据。请使用 README.md 与 review.html；可复算入口为 scripts/negative_review.py。

# Benchmark negative-candidate audit
Run on rabbit05 with the finelap environment. Inputs are frozen runs-source-audit-v2 predictions and manifests. This notebook ranks review candidates; it does not measure false-alarm rates or establish verified training negatives.

In [ ]:
import json,csv,heapq,time
from pathlib import Path
import numpy as np
R=Path.home()/'projects/ov-sed/benchmark'
OUT=R/'negative-review';OUT.mkdir(exist_ok=True)
MODELS=['dasm','finelap','flexsed','mga-clap','openflam']
DATASETS=['desed-validation','realdesed-test','urbansed-test']
all_rows=[];coverage=[]
def overlap(a,b,c,d):return max(0,min(b,d)-max(a,c))
for ds in DATASETS:
 manifest=json.loads((R/'manifests'/f'{ds}.json').read_text())
 for model in MODELS:
  heaps={c:[] for c in manifest['classes']};valid_windows={c:0 for c in heaps};present=0
  for item in manifest['items']:
   path=R/'runs-source-audit-v2'/model/ds/(item['id']+'.npz')
   if not path.exists():continue
   p=np.load(path);scores=p['scores'];times=p['timestamps'];duration=float(times[-1]);events=item['events'];present+=1
   best={}
   for start in range(max(0,int(np.floor(duration-2))+1)):
    end=start+2.;weights=np.maximum(0,np.minimum(times[1:],end)-np.maximum(times[:-1],start))
    if weights.sum()<1.999:continue
    means=(scores*weights[:,None]).sum(0)/weights.sum()
    for j,c in enumerate(p['labels']):
     same=[e for e in events if e[2]==c]
     if any(overlap(start,end,e[0],e[1])>1e-6 for e in same):continue
     valid_windows[c]+=1
     others=sorted({e[2] for e in events if e[2]!=c and overlap(start,end,e[0],e[1])>0})
     near=any(overlap(max(0,start-.5),min(duration,end+.5),e[0],e[1])>0 for e in same)
     kind='near_target_boundary' if near else ('other_labeled_events' if others else 'unlabeled_background')
     row=dict(dataset=ds,model=model,clip_id=item['id'],audio_path=item['path'],onset=float(start),offset=end,target_class=str(c),mean_score=float(means[j]),peak_score=float(scores[weights>0,j].max()),candidate_type=kind,overlapping_labels=others,target_elsewhere=bool(same),review_status='unreviewed')
     if c not in best or row['mean_score']>best[c]['mean_score']:best[c]=row
   for c,row in best.items():
    entry=(row['mean_score'],row['clip_id'],row['onset'],row)
    heap=heaps[c]
    if len(heap)<5:heapq.heappush(heap,entry)
    elif entry[:3]>heap[0][:3]:heapq.heapreplace(heap,entry)
  for c,h in heaps.items():
   for rank,(_,_,_,row) in enumerate(sorted(h,reverse=True),1):row['rank_within_model_dataset_class']=rank;all_rows.append(row)
  coverage.append(dict(dataset=ds,model=model,expected=len(manifest['items']),predictions=present,eligible_negative_windows=valid_windows))
  print(ds,model,present,flush=True)
(OUT/'candidates.json').write_text(json.dumps(all_rows,ensure_ascii=False,indent=2))
with open(OUT/'candidates.csv','w',newline='',encoding='utf-8-sig') as f:
 writer=csv.DictWriter(f,fieldnames=list(all_rows[0]));writer.writeheader()
 for row in all_rows:writer.writerow({**row,'overlapping_labels':' | '.join(row['overlapping_labels'])})
from collections import Counter,defaultdict
counts=Counter(x['candidate_type'] for x in all_rows)
shared=defaultdict(list)
for x in all_rows:shared[(x['dataset'],x['clip_id'],x['target_class'])].append(x)
common=[]
for key,rows in shared.items():
 if len({x['model'] for x in rows})>=3:
  # Require actual overlap of selected 2-second regions; file-only agreement is insufficient.
  for t in sorted({x['onset'] for x in rows}):
   selected=[x for x in rows if x['onset']<=t and x['offset']>=t+1]
   if len({x['model'] for x in selected})>=3:
    common.append(dict(dataset=key[0],clip_id=key[1],target_class=key[2],models=sorted({x['model'] for x in selected}),onset=max(x['onset'] for x in selected),offset=min(x['offset'] for x in selected),overlapping_labels=sorted({c for x in selected for c in x['overlapping_labels']}),audio_path=rows[0]['audio_path']));break
summary=dict(candidate_count=len(all_rows),candidate_types=dict(counts),coverage=coverage,shared_candidates=sorted(common,key=lambda x:-len(x['models'])),note='Top 5 distinct files per model/dataset/class, ranked by 2-second mean score over windows with zero target-label overlap. Not thresholded false positive counts, not comparable error rates. Candidates require listening; absent annotation is not verified acoustic absence. Benchmark data must remain held out.')
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print(json.dumps({k:v for k,v in summary.items() if k!='coverage'},ensure_ascii=False)[:3000],flush=True)
